In [1]:
import pandas as pd
import numpy as np

In [ ]:
orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products__prior.csv")
products = pd.read_csv("data/products.csv")


<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\yashk\AppData\Local\Temp\ipykernel_4740\1017154782.py:1: SyntaxWarning: invalid escape sequence '\M'
  orders = pd.read_csv("D:\ML\ML Projects\MLDiaries06_MarketBasketAnalysis\data\orders.csv")


In [3]:
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [4]:
orders.columns


Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order'],
      dtype='object')

In [5]:
order_products.head()


,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [6]:
products.head()


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [7]:
order_items = (
    order_products
    .merge(products, on="product_id", how="left")
    .merge(orders[["order_id", "user_id", "order_number"]], 
           on="order_id", how="left")
)


In [8]:
order_items.isnull().sum()


order_id             0
product_id           0
add_to_cart_order    0
reordered            0
product_name         0
aisle_id             0
department_id        0
user_id              0
order_number         0
dtype: int64

In [9]:
order_items.head()


,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,user_id,order_number
0,2,33120,1,1,Organic Egg Whites,86,16,202279,3
1,2,28985,2,1,Michigan Organic Kale,83,4,202279,3
2,2,9327,3,0,Garlic Powder,104,13,202279,3
3,2,45918,4,1,Coconut Butter,19,13,202279,3
4,2,30035,5,0,Natural Sweetener,17,13,202279,3


In [10]:
orders["order_number"].describe()


count    3.421083e+06
mean     1.715486e+01
std      1.773316e+01
min      1.000000e+00
25%      5.000000e+00
50%      1.100000e+01
75%      2.300000e+01
max      1.000000e+02
Name: order_number, dtype: float64

In [11]:
order_items = order_items.sort_values(
    by=["user_id", "order_number"]
)


In [12]:
order_baskets = (
    order_items
    .groupby(["user_id", "order_number"])["product_name"]
    .apply(list)
    .reset_index()
)


In [13]:
order_baskets.head(10)


,user_id,order_number,product_name
0,1,1,"[Soda, Organic Unsweetened Vanilla Almond Milk..."
1,1,2,"[Soda, Pistachios, Original Beef Jerky, Bag of..."
2,1,3,"[Soda, Original Beef Jerky, Pistachios, Organi..."
3,1,4,"[Soda, Original Beef Jerky, Pistachios, Organi..."
4,1,5,"[Soda, Original Beef Jerky, Pistachios, Organi..."
5,1,6,"[Soda, Original Beef Jerky, Pistachios, Organi..."
6,1,7,"[Soda, Pistachios, Original Beef Jerky, Organi..."
7,1,8,"[Original Beef Jerky, Soda, Pistachios, Organi..."
8,1,9,"[Organic Half & Half, Zero Calorie Cola, Organ..."
9,1,10,"[Soda, Zero Calorie Cola, Milk Chocolate Almon..."


In [14]:
user_sequences = (
    order_baskets
    .groupby("user_id")["product_name"]
    .apply(list)
)


In [15]:
user_sequences = user_sequences[
    user_sequences.apply(len) >= 2
]


In [16]:
sequences = [
    tuple(tuple(items) for items in user_seq)
    for user_seq in user_sequences.tolist()
]


In [17]:
type(sequences), type(sequences[0]), type(sequences[0][0])


(list, tuple, tuple)

In [18]:
MIN_SUPPORT = 200    
MAX_LEN = 3      


In [19]:
from prefixspan import PrefixSpan

ps = PrefixSpan(sequences)
ps.maxlen = MAX_LEN

patterns = ps.frequent(MIN_SUPPORT)
len(patterns)


60

In [20]:
patterns[:10]


[(277, [('Coconut Water',)]),
 (1189, [('Soda',)]),
 (484, [('Soda',), ('Soda',)]),
 (269, [('Soda',), ('Soda',), ('Soda',)]),
 (236, [('Organic Yellow Onion',)]),
 (606, [('Clementines',)]),
 (265, [('Popcorn',)]),
 (1390, [('Banana',)]),
 (283, [('Banana',), ('Banana',)]),
 (405, [('Smartwater',)])]

In [21]:
journeys = [
    (support, seq)
    for support, seq in patterns
    if len(seq) >= 2
]

len(journeys)


6

In [22]:
for support, seq in journeys[:5]:
    print(f"Support: {support}")
    print(" → ".join([", ".join(step) for step in seq]))
    print("-" * 50)


Support: 484
Soda → Soda
--------------------------------------------------
Support: 269
Soda → Soda → Soda
--------------------------------------------------
Support: 283
Banana → Banana
--------------------------------------------------
Support: 350
Spring Water → Spring Water
--------------------------------------------------
Support: 202
Spring Water → Spring Water → Spring Water
--------------------------------------------------


In [23]:
def remove_consecutive_duplicates(sequence):
    cleaned = []
    prev = None
    for step in sequence:
        step_set = set(step)
        if step_set != prev:
            cleaned.append(step)
        prev = step_set
    return cleaned

sequences_clean = [
    tuple(remove_consecutive_duplicates(seq))
    for seq in sequences
]


In [24]:
sequences_clean[8]


(('Almond Non-Dairy Yogurt Made From Real Almonds Plain Low Fat',
  'Black Peppercorns',
  'Organic Raspberries',
  'Total 2% with Strawberry Lowfat Greek Strained Yogurt',
  'Total 2% Lowfat Greek Strained Yogurt With Blueberry',
  'Total 0% Peach Yogurt',
  'Total 2% Greek Strained Yogurt with Cherry 5.3 oz',
  'Organic Yellow Peaches',
  'Dairy Free Vanilla Coconut Milk',
  'Organic Bunny Fruit Snacks Berry Patch',
  'Organic Iced Oatmeal Cookie ZBar',
  'Organic Summer Strawberry Bunny Fruit Snacks',
  'Organic AppleApple',
  'Backyard Barbeque Potato Chips',
  'Mini Peanut Butter Sandwich Crackers',
  'Apple Cinnamon GoGo Squeez',
  'Organic Stage 3 Pumpkin Cranberry Apple Baby Food',
  'Butternut Squash Pear Stage 2 Baby Food',
  'Baby Food Stage 2 Blueberry Pear & Purple Carrot',
  'Sunny Days Strawberry Snack Bars',
  'Organic Sunny Days Apple Snack Bars',
  'Cinnamon Crunch Apple Chips',
  'Multigrain Pancake & Waffle Mix',
  'Chicken & Apple Breakfast Sausage',
  'Naturals Sa

In [25]:
ps = PrefixSpan(sequences_clean)
ps.maxlen = 3

patterns = ps.frequent(200)

journeys = [
    (support, seq)
    for support, seq in patterns
    if len(seq) >= 2
]

journeys


[(385, [('Soda',), ('Soda',)]),
 (204, [('Banana',), ('Banana',)]),
 (253, [('Spring Water',), ('Spring Water',)]),
 (349, [('Bag of Organic Bananas',), ('Bag of Organic Bananas',)])]

In [26]:
from collections import Counter

item_counts = Counter()
transition_counts = Counter()

for seq in sequences_clean:
    for i in range(len(seq) - 1):
        a = seq[i][0]
        b = seq[i+1][0]
        if a == b:
            transition_counts[a] += 1
        item_counts[a] += 1

confidence = {
    item: transition_counts[item] / item_counts[item]
    for item in transition_counts
}
confidence


{'Soda': 0.2869977791237634,
 'Chipotle Beef & Pork Realstick': 0.21782178217821782,
 'Vanilla Unsweetened Almond Milk': 0.09574468085106383,
 'Antioxidant Infusions Beverage Malawi Mango': 0.21875,
 'Organic Strawberries': 0.13495768340354472,
 'Asparagus': 0.08783505154639175,
 'Original Hummus': 0.14835368109507954,
 'Bag of Organic Bananas': 0.2723365315290333,
 'Whole Milk': 0.26126487498328654,
 'Organic Mung-Bean Sprouts': 0.05429864253393665,
 'Dried Mangos': 0.1568627450980392,
 'Trail Mix': 0.2275132275132275,
 'Organic Baby Spinach': 0.12589084555611751,
 'Natural Lime Flavor Sparkling Mineral Water': 0.2509881422924901,
 'Lime Italian Sparkling Mineral Water': 0.30338983050847457,
 'Chocolate Coconut Milk Beverage': 0.2727272727272727,
 'Unsweetened Premium Iced Tea': 0.17688266199649738,
 'Hard Boiled Eggs': 0.19101123595505617,
 'Clementines': 0.22483279025896072,
 'New Orleans Iced Coffee': 0.2642089093701997,
 'Sparkling Water Grapefruit': 0.2331102613129382,
 'Cold Bre

In [30]:
def recommend_reorder(last_purchased_item, habit_confidence, min_confidence=0.25):
    """
    Habit-based reorder recommender using confidence only
    """
    if last_purchased_item not in habit_confidence:
        return f"No habit data available for '{last_purchased_item}'."
    
    confidence = habit_confidence[last_purchased_item]
    
    if confidence >= min_confidence:
        return (
            f" Recommend reordering: {last_purchased_item} "
            f"(habit confidence = {confidence:.2f})"
        )
    else:
        return f"Weak reorder habit for '{last_purchased_item}' (confidence = {confidence:.2f})"


In [31]:
print(recommend_reorder("Soda", confidence))
print(recommend_reorder("Bag of Organic Bananas", confidence))
print(recommend_reorder("Asparagus", confidence))
print(recommend_reorder("Milk", confidence))


 Recommend reordering: Soda (habit confidence = 0.29)
 Recommend reordering: Bag of Organic Bananas (habit confidence = 0.27)
Weak reorder habit for 'Asparagus' (confidence = 0.09)
Weak reorder habit for 'Milk' (confidence = 0.24)
